# 03 — Silver (typed, conformed)

Shreds every bronze VARIANT into a typed `silver_<entity>` table for all entities, driven by the `(path, alias, type)` maps in `synergy_schemas.SILVER_COLUMNS` (verified against the OpenAPI spec).

> Uses `CREATE OR REPLACE TABLE` (full refresh), which works through Databricks Connect where `MERGE` isn't supported. Inside a workspace cluster you can switch to `MERGE` for incremental loads.

**The conformance key:** `teams.external_id_mlbam` (and `games.external_id_mlbam`) is the MLBAM id, the standard cross-source join key, so this Synergy data lines up with any other MLBAM-keyed data the customer has. It's also the spine of the gold star schema (notebook 05).

In [ ]:
# Job parameter bridge: when run as a Databricks Job task, the bundle's base_parameters arrive as
# notebook widgets. Mirror them into os.environ before the setup cell reads them, so the same
# os.getenv(...) logic works as a job, in a workspace, or locally via .env. dbutils is absent under
# local Databricks Connect, so the except clause handles that case.
import os
for _k in ("UC_CATALOG", "UC_SCHEMA"):
    try:
        _v = dbutils.widgets.get(_k)            # noqa: F821 — dbutils is injected in Databricks
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks Git Folder.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # already present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
from synergy_schemas import SILVER_COLUMNS, PRIMARY_KEYS, ENTITIES

def build_silver(entity: str):
    cols = SILVER_COLUMNS[entity]
    # Some entities land no data on a given run: empty for this credential's org (e.g.
    # practice_training_workout) or a transient source 5xx. The bronze table is then absent, so create an
    # empty, correctly-typed silver shell from SILVER_COLUMNS. Every modeled silver table must always
    # exist so downstream (04 DQ, 05 gold, 06 Genie) resolves it instead of failing on a missing table.
    if not spark.catalog.tableExists(f"{B}.bronze_{entity}"):
        empty = ",\n        ".join(f"cast(null AS {typ}) AS {alias}" for _, alias, typ in cols)
        spark.sql(f"""
            CREATE OR REPLACE TABLE {S}.silver_{entity} AS
            SELECT
            {empty},
            cast(null AS timestamp) AS _ingestion_timestamp
            LIMIT 0
        """)
        print(f"  silver_{entity:26} {'0':>7}  (empty shell — no bronze data)")
        return
    # VARIANT navigation: `data:a:b::type`. Nested objects use `:` (e.g. data:homeTeam:division:id).
    # Clean values cast directly. If an entity has junk values that hard-fail a cast, swap
    # `data:{path}::{typ}` for `try_cast(data:{path}::string AS {typ})` to null bad values instead.
    select = ",\n        ".join(f"data:{path}::{typ} AS {alias}" for path, alias, typ in cols)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {S}.silver_{entity} AS
        SELECT
        {select},
        _ingestion_timestamp
        FROM {B}.bronze_{entity}
    """)
    # Dedup to the natural key (latest ingest wins) so re-pulls converge.
    pk = PRIMARY_KEYS[entity]
    on = " AND ".join(f"a.{k} <=> b.{k}" for k in pk)
    spark.sql(f"""
        CREATE OR REPLACE TABLE {S}.silver_{entity} AS
        SELECT * EXCEPT(_rn) FROM (
          SELECT *, row_number() OVER (PARTITION BY {','.join(pk)} ORDER BY _ingestion_timestamp DESC) _rn
          FROM {S}.silver_{entity}
        ) WHERE _rn = 1
    """)
    print(f"  silver_{entity:26} {spark.table(f'{S}.silver_{entity}').count():>7,} rows  ({len(cols)} cols)")

for e in ENTITIES:
    try:
        build_silver(e["name"])
    except Exception as ex:
        print(f"  silver_{e['name']:26} SKIP/err: {str(ex)[:70]}")

In [ ]:
# sample + the cross-source conformance key
spark.sql(f"SELECT id, name, external_id_mlbam, league_name, division_name FROM {S}.silver_teams LIMIT 5").show(truncate=False)
spark.sql(f"SELECT id, season, game_date, home_team_name, away_team_name, game_level FROM {S}.silver_games LIMIT 5").show(truncate=False)